# Etapa 5l — Familias nuevas de mapas para los vasos finos

**Ronda de mejoras para la entrega final (desvío 14.11 del plan).** Solo desarrollo, la misma validación cruzada por personas. El conjunto de prueba no se usa.

**Motivo.** La limitación principal del algoritmo son los vasos finos: encuentra cerca del 35 % de su línea central. Agregar mapas del mismo tipo ya rinde milésimas (textura, borde asimétrico, mapas contra lesiones), y los otros canales de color no suman (`02d`). Se prueban cuatro familias de mapas que no tenemos, respaldadas por artículos, ordenadas por cuánto atacan el problema de los finos:

| Familia | Artículo | Qué aporta | Mapas |
|---|---|---|---|
| **Simetría de fase** (`enhance.phase_symmetry`) | Kovesi (1999); Läthén, Jonasson y Borga (2010) | Nuestros filtros responden en proporción al contraste; la fase no: un vaso fino tenue da casi lo mismo que uno marcado. Filtros log-Gabor en cuadratura (8 orientaciones, longitud de onda 2 × ancho de cada escala de la etapa 4). | 6: todas las escalas y cada una |
| **B-COSFIRE** (`enhance.bcosfire`) | Azzopardi, Strisciuglio, Vento y Petkov (2015) | Media geométrica de respuestas de diferencia de gaussianas a lo largo de una línea: responde a barras más que a manchas; la versión asimétrica, también a las terminaciones de los vasos. Las dos configuraciones del artículo para DRIVE, llevadas al diámetro del campo, y la simétrica a un tercio de esa escala (la base resuelve vasos unas tres veces más finos que DRIVE). | 3 |
| **Aperturas por caminos** (`enhance.path_openings`) | Talbot y Appleton (2007) | Conservan lo claro que contiene un camino largo que puede curvarse (hasta 45° de una dirección), en lugar de un segmento recto: no pierden los vasos finos tortuosos. Largos de 2 % y 4 % del diámetro, los de la apertura por reconstrucción. | 2 |
| **Autovalores del hessiano** (`enhance.hessian_eigenvalues`) | Niemeijer et al. (2004); Staal et al. (2004) | Frangi resume los dos autovalores en un número con parámetros fijos; sueltos, la regresión aprende la combinación. En las 5 escalas de la etapa 4. | 10 |

Las pruebas de cada familia están en `tests/test_enhance.py` (la de los caminos se compara con una búsqueda directa). Los parámetros son los de los artículos o de criterio; no se ajustaron con datos.

**Candidatas** (sobre el algoritmo actual: Wiener con ruido estimado, relleno de exudados, división, entrenamiento equilibrado por grosor, valor fijo elegido en entrenamiento):
- **A.** Los 28 mapas actuales.
- **B.** A + simetría de fase (34). **C.** A + B-COSFIRE (31). **D.** A + aperturas por caminos (30). **E.** A + autovalores del hessiano (38).
- **F.** A + las cuatro familias (49).

**Medidas**, en las personas de afuera, a resolución original, en el campo de visión de la base: vasos finos (sensibilidad de línea central), Matthews, Dice de línea central y precisión. Descriptivo: por grupo.

**Regla de adopción (fijada antes de correr).** Como estas familias apuntan a los vasos finos, una candidata **cumple** si
1. mejora los vasos finos con un intervalo al 95 % que no incluye el cero, **y**
2. no es inferior a A en Matthews ni en Dice de línea central (límite inferior > −0,03).

Si varias cumplen, gana la de menos mapas entre las que no son inferiores en vasos finos a la que más los mejora (margen −0,03). Si ninguna cumple, queda A. Después de esta ronda viene la selección con L1 sobre los mapas que queden.

**Control:** A tiene que reproducir el Matthews de B en `05k` (0,7525).

In [1]:
import sys
import time
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from src import config as C
from src.data import dev_ids, load_split
from src.evaluation import map_images
from src.pipeline import choose_value, fit_weights
from src.split import person_folds
from src.stats import percentile_interval, person_bootstrap_means

pd.set_option("display.width", 220)
ids = dev_ids()
dev = load_split().set_index("image").loc[ids]
folds = dev["person"].map(person_folds(dev.reset_index(), 5, C.SEED))
MODE = C.SEGMENT_BACKGROUND_MODE
FAM = C.NEW_FEATURE_FAMILIES
SUPER = C.SEGMENT_FEATURES + [n for names in FAM.values() for n in names]
CANDS = {"A. actual": C.SEGMENT_FEATURES,
         "B. + fase": C.SEGMENT_FEATURES + FAM["fase"],
         "C. + B-COSFIRE": C.SEGMENT_FEATURES + FAM["B-COSFIRE"],
         "D. + caminos": C.SEGMENT_FEATURES + FAM["caminos"],
         "E. + hessiano": C.SEGMENT_FEATURES + FAM["hessiano"],
         "F. + todas": SUPER}
COLS = {c: [SUPER.index(n) for n in names] for c, names in CANDS.items()}
ORDER = list(CANDS)
WORKERS = 4
print(MODE, "|", {c: len(v) for c, v in CANDS.items()})

división + relleno de exudados + Wiener con ruido estimado | {'A. actual': 28, 'B. + fase': 34, 'C. + B-COSFIRE': 31, 'D. + caminos': 30, 'E. + hessiano': 38, 'F. + todas': 49}


## Pasada 1: píxeles de entrenamiento con los 49 mapas (cada candidata usa sus columnas)

In [2]:
def pass1(image_id):
    from src.pipeline import training_pixels
    return image_id, training_pixels(image_id, background=MODE, features=SUPER)

t = time.time()
samples_all = dict(map_images(pass1, ids, workers=WORKERS))
print(f"Pasada 1: {time.time() - t:.0f} s")


def subset(c):
    keys_x = ("x", "x_thr")
    return {i: {k: (v[:, COLS[c]] if k in keys_x else v) for k, v in s.items()} for i, s in samples_all.items()}

Pasada 1: 527 s


## Ajuste por vuelta

In [3]:
fit = {}
for c in CANDS:
    sc = subset(c)
    for k in range(5):
        tr = list(folds.index[folds != k])
        w = fit_weights(sc, tr)
        full = np.zeros(1 + len(SUPER))
        full[0] = w[0]
        full[1 + np.array(COLS[c])] = w[1:]
        fit[(k, c)] = (full, choose_value(sc, w, tr))
pd.DataFrame({c: [fit[(k, c)][1] for k in range(5)] for c in CANDS}, index=[f"vuelta {k}" for k in range(5)]).round(4)

,A. actual,B. + fase,C. + B-COSFIRE,D. + caminos,E. + hessiano,F. + todas
vuelta 0,0.8801,0.8938,0.8804,0.8832,0.8658,0.8856
vuelta 1,0.8761,0.8906,0.8758,0.8784,0.8782,0.8817
vuelta 2,0.8785,0.8783,0.8790,0.8817,0.8646,0.8847
vuelta 3,0.8644,0.8799,0.8646,0.8837,0.8667,0.8864
vuelta 4,0.8730,0.9014,0.8732,0.8915,0.8901,0.8946


## Pasada 2: las personas de afuera (los mapas se calculan una vez para las 6 candidatas)

In [4]:
def pass2(image_id):
    import numpy as np
    from src.config import WIDTH_LIMITS_PX
    from src.data import load_fov, load_image, load_vessels
    from src.enhance import upscale
    from src.fov import detect_fov_robust
    from src.metrics import centerline_dice, matthews
    from src.pipeline import feature_scores
    from src.vessels import skeleton_widths
    k = int(folds[image_id])
    rgb = load_image(image_id)
    truth, fov = load_vessels(image_id), load_fov(image_id)
    auto = detect_fov_robust(rgb)
    skel, wd = skeleton_widths(truth & fov)
    thin = skel & (wd <= WIDTH_LIMITS_PX[0])
    probs, _, _ = feature_scores(rgb, [fit[(k, c)][0] for c in ORDER], 2, background=MODE, features=SUPER)
    rows = []
    for c, p in zip(ORDER, probs):
        m = (upscale(p, fov.shape) > fit[(k, c)][1]) & auto
        marked = m & fov
        rows.append({"image": image_id, "candidata": c, "finos": m[thin].mean(), "matthews": matthews(m, truth, fov),
                     "dice_linea_central": centerline_dice(m, truth, fov),
                     "precision": (marked & truth).sum() / max(marked.sum(), 1)})
    return rows

t = time.time()
R = pd.DataFrame([r for rows in map_images(pass2, ids, workers=WORKERS) for r in rows])
R.to_csv(ROOT / "results" / "05l_mapas_nuevos.csv", index=False)
print(f"Pasada 2: {time.time() - t:.0f} s")

Pasada 2: 552 s


## Resultados

In [5]:
MEAS = ("finos", "matthews", "dice_linea_central", "precision")


def diff(new, old, meas):
    w = R.pivot(index="image", columns="candidata", values=meas)
    d = (w[new] - w[old]).to_frame()
    lo, hi = percentile_interval(person_bootstrap_means(d, dev["person"]))
    return d.iloc[:, 0].mean(), lo[0], hi[0], int((d.iloc[:, 0] > 0).sum())


rows = []
for c in ORDER[1:]:
    for meas in MEAS:
        m, lo, hi, n = diff(c, ORDER[0], meas)
        rows.append({"candidata": c, "medida": meas, "diferencia con A": m, "inferior": lo, "superior": hi,
                     "imágenes que mejoran": n})
table = pd.DataFrame(rows).set_index(["candidata", "medida"])
print("Medias:")
display(R.pivot_table(index="candidata", values=list(MEAS)).loc[ORDER, list(MEAS)].round(4))
table.round(4)

Medias:


,finos,matthews,dice_linea_central,precision
candidata,,,,
A. actual,0.3461,0.7525,0.7568,0.8024
B. + fase,0.3479,0.7525,0.7479,0.8205
C. + B-COSFIRE,0.3480,0.7527,0.7583,0.8019
D. + caminos,0.3353,0.7519,0.7546,0.8070
E. + hessiano,0.3537,0.7525,0.7578,0.7998
F. + todas,0.3596,0.7531,0.7505,0.8112


diferencia con A  inferior  superior  imágenes que mejoran
candidata      medida                                                                        
B. + fase      finos                         0.0018   -0.0071    0.0104                    15
               matthews                     -0.0000   -0.0035    0.0038                    17
               dice_linea_central           -0.0088   -0.0107   -0.0070                     0
               precision                     0.0181    0.0136    0.0231                    30
C. + B-COSFIRE finos                         0.0019    0.0005    0.0033                    19
               matthews                      0.0002   -0.0002    0.0008                    15
               dice_linea_central            0.0016    0.0011    0.0021                    26
               precision                    -0.0005   -0.0015    0.0008                     8
D. + caminos   finos                        -0.0108   -0.0157   -0.0061                     0
               matthews                     -0.0006   -0.0020    0.0006                    18
               dice_linea_central           -0.0022   -0.0040   -0.0005                    12
               precision                     0.0046    0.0026    0.0068                    27
E. + hessiano  finos                         0.0076   -0.0014    0.0167                    20
               matthews                     -0.0001   -0.0013    0.0012                    14
               dice_linea_central            0.0010   -0.0013    0.0032                    21
               precision                    -0.0026   -0.0065    0.0013                    14
F. + todas     finos                         0.0135    0.0053    0.0207                    23
               matthews                      0.0006   -0.0017    0.0034                    14
               dice_linea_central           -0.0062   -0.0080   -0.0044                     2
               precision                     0.0088    0.0046    0.0141                    24

In [6]:
meets = [c for c in ORDER[1:]
         if table.loc[(c, "finos"), "inferior"] > 0
         and table.loc[(c, "matthews"), "inferior"] > -C.NONINF_MARGIN_MATTHEWS
         and table.loc[(c, "dice_linea_central"), "inferior"] > -C.NONINF_MARGIN_MATTHEWS]
print("Cumplen:", meets)
if meets:
    best = max(meets, key=lambda c: table.loc[(c, "finos"), "diferencia con A"])
    close = [c for c in meets if c == best or diff(c, best, "finos")[1] > -C.NONINF_MARGIN_MATTHEWS]
    adopt = min(close, key=lambda c: (len(CANDS[c]), ORDER.index(c)))
    print("La que más mejora los finos:", best, "| no inferiores a ella:", close)
else:
    adopt = ORDER[0]
print("→ se adopta:", adopt, f"({len(CANDS[adopt])} mapas)")

Cumplen: ['C. + B-COSFIRE', 'F. + todas']
La que más mejora los finos: F. + todas | no inferiores a ella: ['C. + B-COSFIRE', 'F. + todas']
→ se adopta: C. + B-COSFIRE (31 mapas)


In [7]:
print("Por grupo (media):")
R.join(dev["group"], on="image").pivot_table(index="candidata", columns="group",
                                              values=["finos", "matthews", "precision"]).loc[ORDER].round(3)

Por grupo (media):


finos               matthews               precision              
group              dr      g      h       dr      g      h        dr      g      h
candidata                                                                         
A. actual       0.363  0.333  0.342    0.706  0.755  0.796     0.714  0.773  0.921
B. + fase       0.347  0.338  0.359    0.714  0.756  0.787     0.742  0.789  0.930
C. + B-COSFIRE  0.363  0.334  0.347    0.708  0.754  0.796     0.716  0.771  0.919
D. + caminos    0.353  0.324  0.329    0.707  0.755  0.794     0.719  0.778  0.925
E. + hessiano   0.371  0.339  0.351    0.706  0.755  0.797     0.711  0.770  0.918
F. + todas      0.359  0.347  0.373    0.713  0.755  0.791     0.734  0.777  0.923

## Decisión (escrita después de correr)

**Control:** A reproduce exacto el Matthews de `05k` (0,7525).

| Contra A (personas de afuera) | Vasos finos (puntos) | Matthews | Dice de línea central | Precisión (puntos) |
|---|---|---|---|---|
| B. + fase (34) | +0,2 (−0,7 a +1,0) | 0,0000 | **−0,009** (−0,011 a −0,007), 0 de 30 | **+1,8** (+1,4 a +2,3), 30 de 30 |
| C. + B-COSFIRE (31) | **+0,2 (+0,05 a +0,3)** | +0,0002 | +0,0016 (+0,0011 a +0,0021) | −0,05 |
| D. + caminos (30) | **−1,1** (−1,6 a −0,6), 0 de 30 | −0,0006 | −0,002 | +0,5 |
| E. + hessiano (38) | +0,8 (−0,1 a +1,7) | −0,0001 | +0,001 | −0,3 |
| F. + todas (49) | **+1,35 (+0,5 a +2,1)** | +0,0006 | −0,006 (−0,008 a −0,004) | +0,9 |

**Se adopta C, B-COSFIRE** (31 mapas), **por la regla y con un efecto mínimo.** Cumplen C y F. F es la que más mejora los vasos finos, pero C no es inferior a ella (margen de 3 puntos) y tiene menos mapas. La ganancia de C es de 0,2 puntos de vasos finos, con un límite inferior de 0,05 puntos: es real, pero no le cambia nada a la segmentación. Cuesta unos 17 segundos más por imagen.

**Lo que se aprende de cada familia:**
- **La simetría de fase hace a la segmentación más precisa: +1,8 puntos en las 30 imágenes**, sobre todo en retinopatía, donde la precisión pasa de 0,714 a 0,742 y Matthews de 0,706 a 0,714. Es la única mejora de toda la ronda en el error de retinopatía. A cambio pierde línea central (Dice de línea central −0,009 en las 30 imágenes) y Matthews en sanos (0,796 → 0,787), así que no cumple la regla. Como no depende del contraste, marca los vasos más angostos (más cerca de como los marcaron los expertos en retinopatía), pero deja de encontrar algunos.
- **Las aperturas por caminos empeoran los vasos finos** (−1,1 puntos en las 30 imágenes). Los finos que no encontramos son tenues, no tortuosos, y una apertura por caminos con umbrales no recupera lo que está por debajo del umbral.
- **Los autovalores del hessiano** suman 0,8 puntos de vasos finos, sin llegar a excluir el cero.
- **Todas juntas (F)** suman 1,35 puntos de vasos finos y 0,9 de precisión, pero bajan el Dice de línea central.

**Limitaciones.** El margen de 3 puntos para "no inferior a la mejor" en vasos finos fue amplio: con él, la regla elige una candidata que casi no cambia nada en lugar de la que más mejora. Se respeta porque se fijó antes de correr. La selección con L1 (próxima ronda) parte de los 31 mapas adoptados. Es la sexta ronda decidida con las mismas 30 imágenes. El conjunto de prueba no se usó.